# 모델 구현하기 - CNN Basic

In [2]:
import torch
import torch.nn as nn

In [4]:
inputs = torch.Tensor(1, 1, 28, 28)
print(f"텐서의 크기 : {inputs.shape}")

텐서의 크기 : torch.Size([1, 1, 28, 28])


In [5]:
conv1 = nn.Conv2d(1, 32, 3, padding=1)
print(conv1)

Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))


In [6]:
conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
print(conv2)

Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))


In [7]:
pool = nn.MaxPool2d(2)
print(pool)

MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)


In [8]:
out = conv1(inputs)
print(out.shape)

torch.Size([1, 32, 28, 28])


In [9]:
out = pool(out)
print(out.shape)

torch.Size([1, 32, 14, 14])


In [10]:
out = conv2(out)
print(out.shape)

torch.Size([1, 64, 14, 14])


In [11]:
out = pool(out)
print(out.shape)

torch.Size([1, 64, 7, 7])


In [12]:
out.size(0)

1

In [13]:
out.size(1)

64

In [15]:
out.size(2)

7

In [16]:
out.size(3)

7

In [18]:
out = out.view(out.size(0), -1)
print(out.shape)

torch.Size([1, 3136])


In [19]:
fc = nn.Linear(3136, 10)
out = fc(out)
print(out.shape)

torch.Size([1, 10])


# CNN으로 MNIST 분류하기

In [22]:
import torch
import torchvision.datasets as dataset
import torchvision.transforms as transforms
import torch.nn.init

In [23]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# random seed 고정
torch.manual_seed(777)

if device == "cuda":
    torch.cuda.manual_seed(777)

In [24]:
# Hyperparameter
learning_rate = 0.001
training_epochs = 15
batch_size = 100

In [25]:
# Dataset
mnist_train = dataset.MNIST(root="MNIST_data/",
                            train=True,
                            transform=transforms.ToTensor(),
                            download=True)

mnist_test = dataset.MNIST(root="MNIST_data/",
                           train=False,
                           transform=transforms.ToTensor(),
                           download=True)

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.62MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 170kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 16.0MB/s]


In [26]:
data_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size=batch_size,
                                          shuffle=True,
                                          drop_last=True)

In [29]:
class CNN(torch.nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # 첫번째 layer
        # ImgIn shape=(?, 28, 28, 1)
        #   Conv   -> (? 28, 28, 32)
        #   Pool   -> (? 14, 14, 32)
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(stride=2, kernel_size=2)
        )

        # 두번째 layer
        # ImgIn shape=(?, 14, 14, 32)
        #   Conv   -> (? 14, 14, 64)
        #   Pool   -> (? 7, 7, 64)
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(stride=2, kernel_size=2)
        )

        # Fully Connected layer 7 by 7 by 64 -> 10
        self.fc = torch.nn.Linear(7*7*64, 10, bias=True)

        # FC layer 초기화
        torch.nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

In [30]:
model = CNN().to(device)

In [31]:
criterion = torch.nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [32]:
total_batch = len(data_loader)
print(f"총 배치의 수 : {total_batch}")

총 배치의 수 : 600


In [40]:
for epoch in range(training_epochs):
    avg_cost = 0

    for X, Y in data_loader:
        # 이미지 데이터는 이미 28 by 28 -> 별도 reshape 불필요
        # 레이블 Y는 원-핫 인코딩이 아닌 정수형 클래스 라벨
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad() # optimizer 기울기 초기화
        hypothesis = model(X) # 모델에 X 넣어 예측값 계산 forward
        cost = criterion(hypothesis, Y) # 예측값과 실제값간의 손실 계산
        cost.backward() # 역전파로 기울기 계산
        optimizer.step() # 가중치 업데이트

        avg_cost += cost / total_batch

    print(f"[Epoch: {epoch+1.:>4}] cost = {avg_cost:>.9}")



[Epoch:  1.0] cost = 0.00321334181
[Epoch:  2.0] cost = 0.00116657326
[Epoch:  3.0] cost = 0.00387615152
[Epoch:  4.0] cost = 0.00336503726
[Epoch:  5.0] cost = 0.00171654101
[Epoch:  6.0] cost = 0.00336833182
[Epoch:  7.0] cost = 0.00236956729
[Epoch:  8.0] cost = 0.000575826736
[Epoch:  9.0] cost = 0.000151037646
[Epoch: 10.0] cost = 3.88955887e-05
[Epoch: 11.0] cost = 2.84966645e-05
[Epoch: 12.0] cost = 2.07868034e-05
[Epoch: 13.0] cost = 1.68664228e-05
[Epoch: 14.0] cost = 1.36584385e-05
[Epoch: 15.0] cost = 1.08843842e-05


In [36]:
with torch.no_grad():
    X_test = mnist_test.data.view(len(mnist_test), 1, 28, 28).float().to(device) # test dataset 크기 맞추고 flatten
    Y_test = mnist_test.targets.to(device)

    prediction = model(X_test) # 모델에 테스트 데이터 넣어 예측값 계산
    correct_prediction = torch.argmax(prediction, 1) == Y_test # 예측값과 실제값 비교
    accuracy = correct_prediction.float().mean() # 정확도를 계산하기 위해 일치하는 예측의 평균
    print(f"Accuracy: {accuracy.item()}, {accuracy*100:.2f}%")

Accuracy: 0.9836999773979187, 98.37%


In [42]:
# GPU 메모리 정리
import gc
gc.collect()
torch.cuda.empty_cache()